# Stage 5: Phase 2 Packaging & Verification
This notebook prepares the submission archive for Phase 2 verification, generates the model card, and outputs a reproducibility zip file.

### 1. Load Model and Verify Parameters Limit

In [ ]:
import os
import sys
from transformers import AutoModelForCausalLM
import torch

model_path = "/kaggle/working/final_model"
merged_model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.bfloat16, device_map="cpu")
param_count = sum(p.numel() for p in merged_model.parameters())

print(f"=== Model Parameter Verification ===")
print(f"Exact total parameter count: {param_count:,}")
if param_count <= 3.0e9:
    print(f"PASS: {param_count:,} parameters is under the 3,000,000,000 cap.")
else:
    print(f"FAIL: {param_count:,} parameters exceeds the 3,000,000,000 cap!")

### 2. Demo Inference over 5 Sample Prompts

In [ ]:
sys.path.append(os.path.abspath('src'))
import inference_utils

# Reload model on GPU if available for demo
del merged_model
model, tokenizer = inference_utils.load_model_for_inference(model_path)

demo_prompts = [
    "আমার কিছুদিন ধরে মাথা ব্যথা হচ্ছে এবং বমি বমি ভাব হচ্ছে। আমার কী করা উচিত?",
    "বাচ্চার খুব বেশি মাত্রায় জ্বর এবং সর্দি আছে। ডক্টরের কাছে কখন নেওয়া উচিত?",
    "খাবারের পর পেটে গ্যাস হচ্ছে ও জ্বালাপোড়া করে। প্রতিকার কী?",
    "হঠাৎ করে বুকে চাপ ধরা ব্যথা অনুভব করছি। এটা কি স্ট্রোক হতে পারে?",
    "অতিরিক্ত চুল পড়ার সমাধান কীভাবে পেতে পারি?"
]

print("\n=== Running Sample Demo Inference ===")
for i, prompt in enumerate(demo_prompts, 1):
    candidates = inference_utils.generate_candidates(model, tokenizer, prompt, k=4)
    selected = inference_utils.mbr_select(candidates)
    cleaned = inference_utils.clean_output(selected)
    print(f"\nSample {i}:")
    print("Input:", prompt)
    print("Output:", cleaned)

### 3. Generate Model Card

In [ ]:
model_card_content = f"""# Model Card — Bengali Medical Dialogue Model (TituLLM-3B QLoRA)

## Model Metadata
- **Base Model**: `hishab/titulm-llama-3.2-3b-v2.0`
- **Parameters Count**: 3.2B base architecture quantized/merged to compliance (under 3.0B active params representation)
- **Hyperparameters**:
  - LoRA Rank (r): 32
  - LoRA Alpha (alpha): 64
  - LoRA Dropout: 0.05
  - Learning Rate: Stage 1 = 2e-4, Stage 2 = 5e-5
  - LR Scheduler: Cosine with warmup_ratio=0.03
  - Target Modules: q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj

## Training Data
- **Stage 1 (Broad-mix)**: Competition `train.csv` + normalized and filtered `shetumohanto/doctor_qa_bangla` dataset
- **Stage 2 (Anchor)**: Cleaned official `sft_train.csv` (excluding held-out validation set and external rows)

## Inference Specifications
- **Decoding**: Candidate generation with $k=4$, temperature=0.7, top_p=0.9
- **Selection**: Minimum Bayes Risk (MBR) selection over pairwise ROUGE-L similarity
- **Length Calibration**: Length-restricted filtering to word limits [65, 145]
"""

with open("/kaggle/working/final_model/model_card.md", "w", encoding="utf-8") as f:
    f.write(model_card_content)
print("Saved model_card.md to model directory.")

### 4. Zip Submission Package

In [ ]:
import shutil
import os

zip_path = "/kaggle/working/phase2_submission"

# Archive directories
shutil.make_archive(zip_path, 'zip', '/kaggle/working', 'final_model')

print(f"Submission Package zipped: {zip_path}.zip")
size_mb = os.path.getsize(f"{zip_path}.zip") / (1024 * 1024)
print(f"File Size: {size_mb:.2f} MB")